# Classical column capital detection — evaluation and reproducibility

**Research question:** Can a YOLOv8n detector locate Doric, Ionic, and Corinthian column capitals in architectural photos? This is an educational visual inventory prototype, not an architectural style or heritage assessment.

**Run all in a new Colab runtime:** The default path downloads a frozen public dataset and trained model from the [v1.0 GitHub Release](https://github.com/Silvana-99/classical-capital-detection-yolo/releases/tag/v1.0). No API key or Drive access is required. A fresh run validates the model and makes image evidence. Select a GPU runtime if available; CPU evaluation should also work.

**Original training record:** YOLOv8n, Ultralytics 8.4.163, 30 epochs, batch 16, image size 640, seed 42, Tesla T4; 862 training images and 216 validation images. The original run saved `best.pt` and training curves in the author's Drive; the trained `best.pt` is published in the release. See the training configuration cell near the bottom for how to train again. Validation: precision 0.960, recall 0.939, mAP50 0.975, mAP50–95 0.694 (232 objects in 216 validation images).

**Dataset:** Derived 80/20 three-class subset of [Architecture Segmentation v1 on Roboflow Universe](https://universe.roboflow.com/architecture-dataset/architecture-segmentation/dataset/1). The archive checksum is verified before extraction. Check image rights and version license before any further redistribution.


In [ ]:
import hashlib
import urllib.request
import zipfile
from pathlib import Path
import yaml

DATA_URL = (
    "https://github.com/Silvana-99/classical-capital-detection-yolo/"
    "releases/download/v1.0/classical_capitals_v1_80-20_yolo11.zip"
)
EXPECTED_SHA256 = "e4de1715ff07be6658f256984923a555c5fbb829759f6de1ba14b9a092e268bf"

zip_path = Path("/content/classical_capitals_dataset.zip")
dataset_dir = Path("/content/classical_capitals_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

print("Downloading the public dataset...")
urllib.request.urlretrieve(DATA_URL, zip_path)

sha256 = hashlib.sha256()
with zip_path.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

assert sha256.hexdigest() == EXPECTED_SHA256, "Dataset checksum does not match"
print("Checksum verified.")

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(dataset_dir)

yaml_files = list(dataset_dir.rglob("data.yaml"))
assert yaml_files, "Could not find data.yaml in the downloaded dataset"
cfg_path = yaml_files[0]
dataset_root = cfg_path.parent

cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

print("Dataset ready:", cfg_path)
print(cfg)

In [ ]:
%pip install -q ultralytics==8.4.163

import urllib.request
from pathlib import Path
from ultralytics import YOLO

WEIGHTS_URL = (
    "https://github.com/Silvana-99/classical-capital-detection-yolo/"
    "releases/download/v1.0/best.pt"
)
weights_path = Path("/content/best.pt")
urllib.request.urlretrieve(WEIGHTS_URL, weights_path)

model = YOLO(str(weights_path))
metrics = model.val(
    data=str(cfg_path),
    imgsz=640,
    plots=True,
    project="/content/results",
    name="validation",
    exist_ok=True,
)

print("Precision:", round(metrics.box.mp, 3))
print("Recall:", round(metrics.box.mr, 3))
print("mAP50:", round(metrics.box.map50, 3))
print("mAP50-95:", round(metrics.box.map, 3))
print("Plots saved in /content/results/validation")

In [ ]:
from pathlib import Path

images = sorted((dataset_root / "valid/images").glob("*"))
images = [p for p in images if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

selected = images[::max(1, len(images) // 10)][:10]
assert len(selected) == 10

model.predict(
    source=[str(p) for p in selected],
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/results",
    name="evidence/validation_predictions",
    exist_ok=True,
)

print("Saved 10 predictions to /content/results/evidence/validation_predictions")

In [ ]:
# Save five examples showing the validation ground-truth annotations.
from PIL import Image, ImageDraw

annotation_dir = Path('/content/results/evidence/annotation_examples')
annotation_dir.mkdir(parents=True, exist_ok=True)
label_files = sorted((dataset_root / 'valid/labels').glob('*.txt'))
annotated = 0
for label_path in label_files:
    if not label_path.read_text().strip():
        continue
    image_path = next((dataset_root / 'valid/images' / (label_path.stem + ext)
                       for ext in ('.jpg', '.jpeg', '.png')
                       if (dataset_root / 'valid/images' / (label_path.stem + ext)).exists()), None)
    if image_path is None:
        continue
    im = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(im)
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls, x, y, w, h = map(float, parts[:5])
        x1, y1 = (x-w/2)*im.width, (y-h/2)*im.height
        x2, y2 = (x+w/2)*im.width, (y+h/2)*im.height
        draw.rectangle((x1, y1, x2, y2), outline='yellow', width=4)
        draw.text((max(0, x1), max(0, y1-16)), cfg['names'][int(cls)], fill='yellow')
    im.save(annotation_dir / f'annotation_{annotated+1:02d}.jpg')
    annotated += 1
    if annotated == 5:
        break
assert annotated == 5
print('Saved five labelled examples:', annotation_dir)


In [ ]:
import requests
from pathlib import Path
from google.colab import files
import shutil

new_dir = Path("/content/new_images")
new_dir.mkdir(exist_ok=True)

# Filename, Wikimedia Commons title, expected class
photos = [
    ("new_01_doric.jpg", "File:Old Athena Temple column capitals 02.jpg", "Doric"),
    ("new_02_doric.jpg", "File:Poros Museum - Doric capital.JPG", "Doric"),
    ("new_03_ionic.jpg", "File:Ionic capital Erechtheum, Acropolis, Athens, Greece.jpg", "Ionic"),
    ("new_04_ionic.jpg", "File:Ionic capital (Church of St Catherine, Plaka) on February 25, 2022.jpg", "Ionic"),
    ("new_05_corinthian.jpg", "File:Adriance Memorial Library column capital detail.jpg", "Corinthian"),
]

session = requests.Session()
session.headers.update({"User-Agent": "SilvanaClassicalCapitalsResearch/1.0 (educational project)"})
source_lines = []

for filename, title, expected in photos:
    response = session.get(
        "https://commons.wikimedia.org/w/api.php",
        params={
            "action": "query",
            "format": "json",
            "prop": "imageinfo",
            "iiprop": "url",
            "iiurlwidth": 1200,
            "titles": title,
        },
        timeout=30,
    )
    response.raise_for_status()
    page = next(iter(response.json()["query"]["pages"].values()))
    image_url = page["imageinfo"][0].get("thumburl", page["imageinfo"][0]["url"])

    image_response = session.get(image_url, timeout=60)
    image_response.raise_for_status()
    (new_dir / filename).write_bytes(image_response.content)

    page_url = "https://commons.wikimedia.org/wiki/" + title.replace(" ", "_")
    source_lines.append(f"{filename} | expected: {expected} | {page_url}")
    print("Downloaded:", filename)

sources_file = Path("/content/new_image_sources.txt")
sources_file.write_text("\n".join(source_lines) + "\n")

model.predict(
    source=[str(new_dir / filename) for filename, _, _ in photos],
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/results",
    name="evidence/new_image_predictions",
    exist_ok=True,
)

shutil.copy2(
    sources_file,
    "/content/results/evidence/new_image_predictions/new_image_sources.txt",
)

archive = shutil.make_archive(
    "/content/classical_capitals_results_with_new_images",
    "zip",
    root_dir="/content/results",
)
files.download(archive)

## New-image findings (exploratory)

At confidence 0.25, the model detected the Ionic capital in `new_04` and Corinthian capital in `new_05`. It missed the two Doric capitals and Ionic capital in `new_03`. The five Wikimedia Commons file pages and expected classes are saved in `new_image_sources.txt`; these photos were selected separately from the frozen training dataset, but overlap of underlying scenes has not been independently ruled out. These five examples are illustrative, not a statistically valid estimate of generalization. Reducing the threshold to 0.10 did not recover the three missed detections. Individual author and license details are on each linked source page. The prediction screenshots are adaptations of the respective source images.


In [ ]:
# OPTIONAL: full retraining. Default remains False so Run all is quick and keyless.
FULL_TRAIN = False
if FULL_TRAIN:
    import torch
    trainer = YOLO('yolov8n.pt')
    train_results = trainer.train(
        data=str(cfg_path), epochs=30, batch=16, imgsz=640,
        device=0 if torch.cuda.is_available() else 'cpu', seed=42,
        plots=True, save_period=5, project='/content/results',
        name='retraining_30_epochs', exist_ok=True,
    )
    print('Full training output:', train_results.save_dir)
else:
    print('Full 30-epoch retraining skipped; the released best.pt was evaluated above.')


In [ ]:
# Optional: download a ZIP of the evidence produced by this run.
# Set DOWNLOAD_RESULTS = True when you want a browser download.
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    archive = shutil.make_archive('/content/classical_capitals_results', 'zip', root_dir='/content/results')
    files.download(archive)
else:
    print('Results are in /content/results; set DOWNLOAD_RESULTS=True to download them.')
